# Plot

In [ ]:
library(readr)
library(purrr)
library(dplyr)
library(stringr)
library(fuzzyjoin)
library(ggplot2)
library(dplyr)
library(tidyverse)
library(Matrix)
library(reshape2)
library(RColorBrewer)
library(dplyr)
library(rstatix)

In [ ]:
options(repr.matrix.max.cols = Inf,  # show all columns
        repr.matrix.max.rows = 200)  # adjust rows as you like

## 0. Plot parameters

In [ ]:
out_dir = "/ceph.groups/mshahbazi.grp/rsakata/Figures/siRNA/imaris_plots"

analysis_summary_files = c(
"/ceph.groups/mshahbazi.grp/rsakata/EXP93/Imaris/output/EXP93_analysis_summary.csv",
"/ceph.groups/mshahbazi.grp/rsakata/EXP95/Imaris/output/EXP95_analysis_summary.csv"
)


In [ ]:
FONT.SIZE <- 7
LABEL.FONT.SIZE <- 7
w <- 2 
h <- 2.5
LINE.W <- 0.5/2.141959

# Set geom defaults globally
update_geom_defaults("line",      list(linewidth = LINE.W))
update_geom_defaults("errorbar",  list(linewidth = LINE.W))
#update_geom_defaults("point",     list(size = LINE.W, stroke = LINE.W))

settheme <- theme_minimal() + 
  theme(
    text = element_text(family = "sans"), 
    panel.background = element_blank(),
    panel.grid.major = element_blank(), 
    panel.grid.minor = element_blank(),
    plot.background = element_blank(),
    axis.ticks = element_line(colour = "black", linewidth = LINE.W),
    axis.ticks.length = unit(0.1, "cm"), 
    axis.line = element_line(linewidth = LINE.W, colour = "black"),
    axis.title = element_text(size = FONT.SIZE),
    axis.text = element_text(colour = "black", size = FONT.SIZE),
    strip.text = element_text(size = FONT.SIZE), 
    strip.text.y.left = element_text(angle = 0, hjust = 1, size = FONT.SIZE),
    legend.position = "right",
    legend.title = element_text(size = FONT.SIZE), 
    legend.text = element_text(size = FONT.SIZE),
    legend.key.size = unit(0.3, "cm"),
    axis.text.x = element_text(colour = "black", angle = 0, size = LABEL.FONT.SIZE),
    title = element_text(size = FONT.SIZE) 
  )


In [ ]:
col_aneu = c("euploid"= "#D4D1B3","monosomy"="#109E9D","trisomy"="#F26B3B","complex"= "#886DB0")

col_condition_2 = c("control" = "#285F62", 
               "reversine" = "#CA4F33", 
               "mosaic"= "#E2A557")

col_condition = c("G_R"= "#5E5E5E","Grev_R"="#86AB30","Rrev_G"="#EB5951", "Grev_Rrev"="#F0A329")

col_GFP_group = c("pos"= "#86AB30","neg"="#8d8d8dff")

col_RFP_group = c("pos"= "#EB5951","neg"="#8d8d8dff")

col_ECAD_group = c("pos"="#B165A0", "neg"="#8d8d8dff")


col_GATA3 = "#489C9C"
col_NANOG = "#EA9542"
col_neg    = "#8d8d8dff"

col_GFP = "#86AB30"
col_RFP = "#EB5951"

## 1. Extract summary files

In [ ]:
# Read and combine all files into one dataframe
merged_df <- analysis_summary_files %>%
  map_dfr(read_csv)

In [ ]:
head(merged_df)

In [ ]:
tbl <- merged_df %>%
  group_by(sample, sample_name, exp) %>%
  summarise(n_images = n_distinct(image), .groups = "drop") %>%
  arrange(sample)  # optional

tbl

In [ ]:
unique(merged_df$siRNA)

In [ ]:
unique(merged_df$ECAD)

## 2. Preprocess

### E) %pos marker

In [ ]:
plot_pct_bar_points <- function(
  data,                               # e.g., summary_df
  pct = pct_GATA3,                    # <-- column with % values to plot
  sample = sample,               # sample/category column
  condition = condition,              # grouping/fill column
  out_dir,                    # folder to save (optional)
  title = NULL,                       # default built from pct col name if NULL
  palette = NULL,                     # named vector for fill
  w = 4, h = 1.5,
  y_max = 110,
  y_ticks = 5,
  bar_width = 0.6,
  point_size = 1.1,
  point_alpha = 0.7,
  jitter_width = 0.05
) {
  pct      <- enquo(pct)
  sample   <- enquo(sample)
  condition<- enquo(condition)

  # default title from pct column name if not supplied
  if (is.null(title)) {
    title <- paste0(as_label(pct), "+")
  }

  # per-sample means (by condition) for the chosen pct column
  means_df <- data %>%
    group_by(!!sample, !!condition) %>%
    summarise(mean_pct = mean(!!pct, na.rm = TRUE), .groups = "drop")

  # build plot (reverse sample order, flip coords)
  p <- ggplot(data, aes(x = fct_rev(!!sample), y = !!pct)) +
    geom_col(
      data = means_df,
      aes(y = mean_pct, fill = !!condition),
      width = bar_width
    ) +
    geom_point(
      size = point_size, alpha = point_alpha,
      position = position_jitter(width = jitter_width),
      na.rm = TRUE
    ) +
    labs(x = "", y = "% positive", title = title, fill = rlang::as_name(condition)) +
    settheme +
    scale_y_continuous(limits = c(0, y_max), expand = c(0, 0),
                       breaks = scales::pretty_breaks(y_ticks)) +
    coord_flip()

  if (!is.null(palette)) {
    p <- p + scale_fill_manual(values = palette)
  }

  safe_title <- gsub("[^[:alnum:]_\\-]+","_", title)
  ggplot2::ggsave(file.path(out_dir, sprintf("%s.pdf", safe_title)),
                  plot = p, width = w, height = h)
  options(repr.plot.width=w, repr.plot.height=h)
  p
}

In [ ]:
plot_pct_bar_points <- function(
  data,
  pct = pct_GATA3,
  sample = sample_name,
  condition = exp,
  out_dir,
  title = NULL,
  palette = NULL,
  w = 4,
  h = 1.5,
  y_max = 110,
  y_ticks = 5,
  bar_width = 0.6,
  dodge_width = 0.7,
  point_size = 1.1,
  point_alpha = 0.7,
  jitter_width = 0.05
) {
  pct       <- enquo(pct)
  sample    <- enquo(sample)
  condition <- enquo(condition)

  if (is.null(title)) {
    title <- paste0(as_label(pct), "+")
  }

  # Mean across images for each sample × experiment
  means_df <- data %>%
    group_by(!!sample, !!condition) %>%
    summarise(
      mean_pct = mean(!!pct, na.rm = TRUE),
      .groups = "drop"
    )

  p <- ggplot(
    data,
    aes(
      x = forcats::fct_rev(!!sample),
      y = !!pct,
      fill = !!condition,
      group = !!condition
    )
  ) +
    geom_col(
      data = means_df,
      aes(y = mean_pct),
      width = bar_width,
      position = position_dodge(width = dodge_width),
      alpha = 0.6
    ) +
    geom_point(
      aes(colour = !!condition),
      size = point_size,
      alpha = point_alpha,
      position = position_jitterdodge(
        jitter.width = jitter_width,
        dodge.width = dodge_width
      ),
      na.rm = TRUE
    ) +
    labs(
      x = "",
      y = "% positive",
      title = title,
      fill = as_label(condition),
      colour = as_label(condition)
    ) +
    settheme +
    scale_y_continuous(
      limits = c(0, y_max),
      expand = c(0, 0),
      breaks = scales::pretty_breaks(n = y_ticks)
    ) +
    coord_flip()

  if (!is.null(palette)) {
    p <- p +
      scale_fill_manual(values = palette) +
      scale_colour_manual(values = palette)
  }

  safe_title <- gsub("[^[:alnum:]_\\-]+", "_", title)

  ggplot2::ggsave(
    file.path(out_dir, paste0(safe_title, ".pdf")),
    plot = p,
    width = w,
    height = h
  )

  options(
    repr.plot.width = w,
    repr.plot.height = h
  )

  p
}

In [ ]:
summary_df <- merged_df %>%
  group_by(exp, image, sample_name) %>%
  summarise(
    n = n(),
    
    # Calculate simple percentages based on your existing boolean columns
    pct_GATA3 = 100 * mean(GATA3pos, na.rm = TRUE),
    pct_NANOG = 100 * mean(NANOGpos, na.rm = TRUE),
    #pct_mcherry = 100 * mean(mcherrypos, na.rm = TRUE),
    #pct_GFP   = 100 * mean(GFPpos,   na.rm = TRUE),
    
    # Calculate negative cells (None of the markers are positive)
    pct_negative = 100 * mean(!( NANOGpos | GATA3pos), na.rm = TRUE),
    
    # Calculate double positives (Both markers are positive)
    pct_double_GATA3_NANOG = 100 * mean(GATA3pos & NANOGpos, na.rm = TRUE),
    
    .groups = "drop"
  )


In [ ]:
head(summary_df)

In [ ]:
# Plot %GATA3+
plot_pct_bar_points( data = summary_df, pct = pct_GATA3, sample = sample_name, condition = exp, out_dir = out_dir,
  title = "GATA3+"
)
                    
# Plot %NANOG+
plot_pct_bar_points(summary_df, pct = pct_NANOG, out_dir = out_dir,
                    title = "E_pctNANOG+")

# Plot %mcherry+
#plot_pct_bar_points(summary_df, pct = pct_mcherry,out_dir = out_dir,
#                    title = "E_pctmcherry+", palette =  c("Control"= col_RFP,"Reversine"= col_RFP))

# Plot %GFP+
#plot_pct_bar_points(summary_df, pct = pct_GFP,out_dir = out_dir,
#                    title = "E_pctGFP+", y_max = 40, palette =  c("Control"= col_GFP,"Reversine"= col_GFP))

# Plot % neg
plot_pct_bar_points(summary_df, pct = pct_negative, condition = exp, out_dir = out_dir, title = "E_pctnegative")



In [ ]:
head(merged_df)

### plot by ECAD pos or negative

In [ ]:
summary_df <- merged_df %>%
  group_by(image, sample_name, ECAD, siRNA) %>%
  summarise(
    n = n(),
    
    # Calculate simple percentages based on your existing boolean columns
    pct_GATA3 = 100 * mean(GATA3pos, na.rm = TRUE),
    pct_NANOG = 100 * mean(NANOGpos, na.rm = TRUE),
    #pct_mcherry = 100 * mean(mcherrypos, na.rm = TRUE),
    #pct_GFP   = 100 * mean(GFPpos,   na.rm = TRUE),
    
    # Calculate negative cells (None of the markers are positive)
    pct_negative = 100 * mean(!( NANOGpos | GATA3pos), na.rm = TRUE),
    
    # Calculate double positives (Both markers are positive)
    pct_double_GATA3_NANOG = 100 * mean(GATA3pos & NANOGpos, na.rm = TRUE),
    
    .groups = "drop"
  )

summary_df <- summary_df %>%
  mutate(
    ECAD = factor(ECAD, levels = c("pos", "neg"))
  )

summary_df_sub = summary_df |>
                 filter(siRNA == "ecad")

head(summary_df_sub)

In [ ]:
title = "GATA3_ECADgroup"
w <- 2
h <- 2
options(repr.plot.width=w, repr.plot.height=h)

p = ggplot(summary_df_sub, aes(x = sample_name, y =pct_GATA3 , group= ECAD)) +  # dots for each file
    stat_summary( aes(fill = ECAD), 
      fun = mean, 
      geom = "bar",
      position = position_dodge(width = 0.75),
      alpha = 0.6, width = 0.6) +   # error bars
    geom_jitter(
      aes(fill = ECAD),
      position = position_jitterdodge(jitter.width = 0.15, dodge.width = 0.75),
      size = 0.5, alpha = 0.8, color = "grey30"
    )  +  # average bar
    stat_summary(
      fun.data = mean_se, 
      geom = "errorbar",
      position = position_dodge(width = 0.75),
      width = 0.2, 
      color = "black")+
    labs(
      title = title,
      y = "%GATA3",
      x = "condition"
    )+ settheme+
    theme(
      axis.text.x = element_text(angle = 45, hjust = 1)
    ) +
      scale_y_continuous(limits = c(0, 100), expand = c(0, 0))+
  scale_fill_manual(
    values = col_ECAD_group,
    name = "ECAD"
  ) 


ggsave(file.path(out_dir, sprintf("%s.pdf", title)),
                plot = p, width = w, height = h)

p

In [ ]:
title = "NANOG_ECADgroup"
w <- 2
h <- 2
options(repr.plot.width=w, repr.plot.height=h)

p = ggplot(summary_df_sub, aes(x = sample_name, y =pct_NANOG , group= ECAD)) +  # dots for each file
    stat_summary( aes(fill = ECAD), 
      fun = mean, 
      geom = "bar",
      position = position_dodge(width = 0.75),
      alpha = 0.6, width = 0.6) +   # error bars
    geom_jitter(
      aes(fill = ECAD),
      position = position_jitterdodge(jitter.width = 0.15, dodge.width = 0.75),
      size = 0.5, alpha = 0.8, color = "grey30"
    )  +  # average bar
    stat_summary(
      fun.data = mean_se, 
      geom = "errorbar",
      position = position_dodge(width = 0.75),
      width = 0.2, 
      color = "black")+
    labs(
      title = title,
      y = "%NANOG+",
      x = "condition"
    )+ settheme+
    theme(
      axis.text.x = element_text(angle = 45, hjust = 1)
    ) +
      scale_y_continuous(limits = c(0, 110), expand = c(0, 0))+
  scale_fill_manual(
    values =col_ECAD_group,
    name = "ECAD"
  ) 


ggsave(file.path(out_dir, sprintf("%s.pdf", title)),
                plot = p, width = w, height = h)

p

In [ ]:
title = "doubleneg_ECADgroup"
w <- 2
h <- 2
options(repr.plot.width=w, repr.plot.height=h)

p = ggplot(summary_df_sub, aes(x = sample_name, y =pct_negative , group= ECAD)) +  # dots for each file
    stat_summary( aes(fill = ECAD), 
      fun = mean, 
      geom = "bar",
      position = position_dodge(width = 0.75),
      alpha = 0.6, width = 0.6) +   # error bars
    geom_jitter(
      aes(fill = ECAD),
      position = position_jitterdodge(jitter.width = 0.15, dodge.width = 0.75),
      size = 0.5, alpha = 0.8, color = "grey30"
    )  +  # average bar
    stat_summary(
      fun.data = mean_se, 
      geom = "errorbar",
      position = position_dodge(width = 0.75),
      width = 0.2, 
      color = "black")+
    labs(
      title = title,
      y = "%double negative+",
      x = "condition"
    )+ settheme+
    theme(
      axis.text.x = element_text(angle = 45, hjust = 1)
    ) +
      scale_y_continuous(limits = c(0, 110), expand = c(0, 0))+
  scale_fill_manual(
    values = col_ECAD_group,
    name = "ECAD"
  ) 


ggsave(file.path(out_dir, sprintf("%s.pdf", title)),
                plot = p, width = w, height = h)

p

In [ ]:
title = "doubleneg_ECADgroup"
w <- 1.5
h <- 1.6
options(repr.plot.width=w, repr.plot.height=h)

p = ggplot(summary_df_sub, aes(x = ECAD, y =pct_negative , group= ECAD)) +  # dots for each file
    stat_summary( aes(fill = ECAD), 
      fun = mean, 
      geom = "bar",
      position = position_dodge(width = 0.8),
      alpha = 0.6, width = 0.75, fill = "grey80") +   # error bars
    geom_jitter(
      aes(color = ECAD),
      position = position_jitterdodge(jitter.width = 0.15, dodge.width = 0.75),
      size = 0.5, alpha = 0.8
    )  +  # average bar
    stat_summary(
      fun.data = mean_se, 
      geom = "errorbar",
      position = position_dodge(width = 0.75),
      width = 0.2, 
      color = "black")+
    labs(
      title = title,
      y = "%double negative+",
      x = "condition"
    )+ settheme+
    theme(
      axis.text.x = element_text(angle = 45, hjust = 1)
    ) +
      scale_y_continuous(limits = c(0, 110), expand = c(0, 0))+
  scale_color_manual(
    values = col_ECAD_group,
    name = "ECAD"
  ) 


ggsave(file.path(out_dir, sprintf("%s.pdf", title)),
                plot = p, width = w, height = h)

p

In [ ]:
head(summary_df_sub%>%
    group_by("sample_name"))

In [ ]:
check_test <- function(data, group_var = "ECAD", value_var = "pct_negative",
                        conditions = c("neg", "pos"), state = "sample_name", alpha = 0.05) {

  d <- data %>%
    filter(.data[[group_var]] %in% conditions) %>%
    droplevels()

  d %>%
    group_by(.data[[state]]) %>%
    group_modify(~ {
      g <- split(.x[[value_var]], .x[[group_var]])
      g <- g[conditions]                       # keep the two groups in order

      # need at least 3 non-NA points per group for Shapiro
      n_ok <- all(sapply(g, function(x) sum(!is.na(x)) >= 3))

      if (!n_ok) {
        return(tibble(
          n1 = sum(!is.na(g[[1]])), n2 = sum(!is.na(g[[2]])),
          shapiro_p1 = NA_real_, shapiro_p2 = NA_real_,
          levene_p = NA_real_, normal = NA,
          recommended = "too few points (use Wilcoxon / be cautious)"
        ))
      }

      # normality per group
      sp1 <- shapiro.test(g[[1]])$p.value
      sp2 <- shapiro.test(g[[2]])$p.value
      normal <- (sp1 > alpha) & (sp2 > alpha)

      # equal-variance check (F-test; swap for car::leveneTest if preferred)
      var_p <- tryCatch(var.test(g[[1]], g[[2]])$p.value, error = function(e) NA_real_)

      rec <- if (normal) {
        if (!is.na(var_p) && var_p > alpha) "Student t-test (var.equal = TRUE)"
        else "Welch t-test"
      } else {
        "Wilcoxon rank-sum test"
      }

      tibble(
        n1 = sum(!is.na(g[[1]])), n2 = sum(!is.na(g[[2]])),
        shapiro_p1 = sp1, shapiro_p2 = sp2,
        levene_p = var_p, normal = normal,
        recommended = rec
      )
    }) %>%
    ungroup()
}

# usage
check_test(summary_df_sub)

In [ ]:
summary_df %>%
  #group_by(sample_name) %>%
  t_test(
    pct_negative ~ ECAD,
    p.adjust.method = "BH"
  ) 

In [ ]:
wilcox_res <- summary_df  %>%
  wilcox_test(
   pct_negative ~ ECAD
  ) %>%
  mutate(
    p_use = if ("p.adj" %in% names(.)) p.adj else p,   # fall back to raw p
    stars = case_when(
      p_use < 0.0001 ~ "****",
      p_use < 0.001  ~ "***",
      p_use < 0.01   ~ "**",
      p_use < 0.05   ~ "*",
      TRUE           ~ "ns"
    )
  )

wilcox_res

In [ ]:
title = "GATA3_ECADgroup"
w <- 1.5
h <- 1.6
options(repr.plot.width=w, repr.plot.height=h)

p = ggplot(summary_df_sub, aes(x = ECAD, y =pct_GATA3 , group= ECAD)) +  # dots for each file
    stat_summary( aes(fill = ECAD), 
      fun = mean, 
      geom = "bar",
      position = position_dodge(width = 0.8),
      alpha = 0.6, width = 0.75, fill = "grey80") +   # error bars
    geom_jitter(
      aes(color = ECAD),
      position = position_jitterdodge(jitter.width = 0.15, dodge.width = 0.75),
      size = 0.5, alpha = 0.8
    )  +  # average bar
    stat_summary(
      fun.data = mean_se, 
      geom = "errorbar",
      position = position_dodge(width = 0.75),
      width = 0.2, 
      color = "black")+
    labs(
      title = title,
      y = "%GATA3+",
      x = "condition"
    )+ settheme+
    theme(
      axis.text.x = element_text(angle = 45, hjust = 1)
    ) +
      scale_y_continuous(limits = c(0, 110), expand = c(0, 0))+
  scale_color_manual(
    values = col_ECAD_group,
    name = "ECAD"
  ) 


ggsave(file.path(out_dir, sprintf("%s.pdf", title)),
                plot = p, width = w, height = h)

p

In [ ]:
title = "NANOG_ECADgroup"
w <- 1.5
h <- 1.6
options(repr.plot.width=w, repr.plot.height=h)

p = ggplot(summary_df_sub, aes(x = ECAD, y =pct_NANOG , group= ECAD)) +  # dots for each file
    stat_summary( aes(fill = ECAD), 
      fun = mean, 
      geom = "bar",
      position = position_dodge(width = 0.8),
      alpha = 0.6, width = 0.75, fill = "grey80") +   # error bars
    geom_jitter(
      aes(color = ECAD),
      position = position_jitterdodge(jitter.width = 0.15, dodge.width = 0.75),
      size = 0.5, alpha = 0.8
    )  +  # average bar
    stat_summary(
      fun.data = mean_se, 
      geom = "errorbar",
      position = position_dodge(width = 0.75),
      width = 0.2, 
      color = "black")+
    labs(
      title = title,
      y = "%NANOG+",
      x = "condition"
    )+ settheme+
    theme(
      axis.text.x = element_text(angle = 45, hjust = 1)
    ) +
      scale_y_continuous(limits = c(0, 110), expand = c(0, 0))+
  scale_color_manual(
    values = col_ECAD_group,
    name = "ECAD"
  ) 


ggsave(file.path(out_dir, sprintf("%s.pdf", title)),
                plot = p, width = w, height = h)

p

In [ ]:
out_dir